# AI-Powered Flood Risk Prediction
# Logistic Regression
## Manuela Munoz Ramirez

Trains a Logistic Regression model to classify flood risk at Murray Bridge using real
historical river level data from the Murray–Darling Basin Authority (MDBA).

## 1. Load real data


In [17]:
import common
import joblib
from sklearn.linear_model import LogisticRegression

df = common.load_data()

print("Rows:", len(df))
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
df.head()

Rows: 6364
Date range: 2009-01-08 09:00:00 to 2026-07-02 09:00:00


,datetime,water_level_m,conductivity,water_temp_c
0,2009-01-08 09:00:00,-0.584,680.492,22.788
1,2009-01-09 09:00:00,-0.640,682.482,22.719
2,2009-01-10 09:00:00,-0.662,685.370,22.657
3,2009-01-11 09:00:00,-0.637,691.286,22.730
4,2009-01-12 09:00:00,-0.649,691.175,22.737


## 2. Feature engineering


In [18]:
X, y = common.build_features(df)

print("Risk threshold (m):", round(common.risk_threshold(df), 3))
print("Rows with features:", len(X))
y.value_counts()

Risk threshold (m): 0.806
Rows with features: 6357


high_risk
0    5083
1    1274
Name: count, dtype: int64

## 3. Train/test split


In [19]:
X_train, X_test, y_train, y_test = common.chronological_split(X, y)

print("Train:", len(X_train), "| Test:", len(X_test))

Train: 5403 | Test: 954


## 4. Train Logistic Regression


In [20]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000)
)
model_lr.fit(X_train, y_train)

pred_lr = model_lr.predict(X_test)
prob_lr = model_lr.predict_proba(X_test)[:, 1]

results = [
    common.evaluate("Logistic Regression", y_test, pred_lr, prob_lr),
    common.persistence_baseline(df, y_test),
]
common.comparison_table(results)

,F1,MCC,RMSE,Brier,NSE
model,,,,,
Logistic Regression,0.800,0.741,0.3,0.09,0.505
Persistence baseline,0.796,0.732,NaN,NaN,NaN


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, pred_lr, digits=3))
print("Confusion matrix (rows = actual):")
print(confusion_matrix(y_test, pred_lr))

              precision    recall  f1-score   support

           0      0.981     0.868     0.921       726
           1      0.692     0.947     0.800       228

    accuracy                          0.887       954
   macro avg      0.837     0.908     0.861       954
weighted avg      0.912     0.887     0.892       954

Confusion matrix (rows = actual):
[[630  96]
 [ 12 216]]


## 5. Save trained model

In [22]:
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(model_lr, "../models/logistic_regression.joblib")
print("Model saved to models/logistic_regression.joblib")

Model saved to models/logistic_regression.joblib
